# Q2 - English -> Urdu Transformer (self-contained Kaggle notebook)

All source code is embedded below via `%%writefile`. Just **Run All**.

**Before running:** right sidebar -> Session options -> Accelerator -> **GPU T4 x2** (or P100).

Training takes ~25 min on T4 x2. BLEU is logged every epoch.

In [ ]:
# 1) Download the parallel corpus into /kaggle/working/data
!mkdir -p /kaggle/working/data
!kaggle datasets download -d zainuddin123/parallel-corpus-for-english-urdu-language -p /kaggle/working/data --unzip
!ls /kaggle/working/data

In [ ]:
%%writefile transformer_model.py
"""
Transformer encoder-decoder (Vaswani et al. 2017) for English -> Urdu MT.

Everything is implemented from scratch in PyTorch (no nn.Transformer shortcut)
so the architecture can be inspected component-by-component during the demo.
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding (eq. from the paper)."""
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        B, Lq, _ = q.shape
        Lk = k.size(1)
        Q = self.q(q).view(B, Lq, self.n_heads, self.d_k).transpose(1, 2)
        K = self.k(k).view(B, Lk, self.n_heads, self.d_k).transpose(1, 2)
        V = self.v(v).view(B, Lk, self.n_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            # mask: broadcastable to (B, h, Lq, Lk); True/1 = keep
            scores = scores.masked_fill(mask == 0, float("-inf"))
        attn = self.drop(F.softmax(scores, dim=-1))
        ctx = torch.matmul(attn, V)
        ctx = ctx.transpose(1, 2).contiguous().view(B, Lq, self.d_model)
        return self.out(ctx)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)


class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.ln1 = nn.LayerNorm(d_model); self.ln2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        x = self.ln1(x + self.drop(self.self_attn(x, x, x, src_mask)))
        x = self.ln2(x + self.drop(self.ff(x)))
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.ln1 = nn.LayerNorm(d_model); self.ln2 = nn.LayerNorm(d_model); self.ln3 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, enc, src_mask, tgt_mask):
        x = self.ln1(x + self.drop(self.self_attn(x, x, x, tgt_mask)))
        x = self.ln2(x + self.drop(self.cross_attn(x, enc, enc, src_mask)))
        x = self.ln3(x + self.drop(self.ff(x)))
        return x


class Transformer(nn.Module):
    """Standard encoder-decoder Transformer."""
    def __init__(self, src_vocab, tgt_vocab, d_model=256, n_heads=8,
                 n_enc=4, n_dec=4, d_ff=1024, dropout=0.1, max_len=256, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx
        self.src_emb = nn.Embedding(src_vocab, d_model, padding_idx=pad_idx)
        self.tgt_emb = nn.Embedding(tgt_vocab, d_model, padding_idx=pad_idx)
        self.pos = PositionalEncoding(d_model, max_len)
        self.drop = nn.Dropout(dropout)
        self.enc_layers = nn.ModuleList(
            [EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_enc)])
        self.dec_layers = nn.ModuleList(
            [DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_dec)])
        self.proj = nn.Linear(d_model, tgt_vocab)
        self.d_model = d_model

    def make_src_mask(self, src):
        # (B, 1, 1, S) -> broadcasts to (B, h, L, S)
        return (src != self.pad_idx).unsqueeze(1).unsqueeze(2)

    def make_tgt_mask(self, tgt):
        B, T = tgt.shape
        pad_mask = (tgt != self.pad_idx).unsqueeze(1).unsqueeze(2)          # (B,1,1,T)
        causal = torch.tril(torch.ones(T, T, device=tgt.device, dtype=torch.bool))
        return pad_mask & causal.unsqueeze(0).unsqueeze(0)                  # (B,1,T,T)

    def encode(self, src, src_mask):
        x = self.drop(self.pos(self.src_emb(src) * math.sqrt(self.d_model)))
        for layer in self.enc_layers:
            x = layer(x, src_mask)
        return x

    def decode(self, tgt, enc, src_mask, tgt_mask):
        x = self.drop(self.pos(self.tgt_emb(tgt) * math.sqrt(self.d_model)))
        for layer in self.dec_layers:
            x = layer(x, enc, src_mask, tgt_mask)
        return self.proj(x)

    def forward(self, src, tgt):
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt)
        enc = self.encode(src, src_mask)
        return self.decode(tgt, enc, src_mask, tgt_mask)


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


if __name__ == "__main__":
    m = Transformer(1000, 1000)
    src = torch.randint(1, 1000, (2, 10))
    tgt = torch.randint(1, 1000, (2, 12))
    out = m(src, tgt)
    print("out:", out.shape, "params:", count_params(m))


In [ ]:
%%writefile data_utils.py
"""
Data + BPE tokenizer utilities for the English-Urdu corpus.

The 'parallel-corpus-for-english-urdu-language' Kaggle dataset ships as CSV
or .txt files. We accept either shape and normalise to two aligned lists.
"""
import os
import random
import re
from pathlib import Path

import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace


SPECIAL = ["<pad>", "<bos>", "<eos>", "<unk>"]
PAD_IDX, BOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3


def _clean(s):
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s


def load_pairs(root):
    """Look for a CSV with 'English','Urdu' columns, else two parallel .txt files."""
    root = Path(root)
    csvs = list(root.glob("*.csv"))
    pairs = []

    if csvs:
        import csv
        with open(csvs[0], "r", encoding="utf-8", errors="ignore") as f:
            rdr = csv.DictReader(f)
            # tolerate different column names
            eng_key = next((k for k in rdr.fieldnames if "eng" in k.lower()), rdr.fieldnames[0])
            urd_key = next((k for k in rdr.fieldnames if "urd" in k.lower()), rdr.fieldnames[1])
            for row in rdr:
                en, ur = _clean(row.get(eng_key, "")), _clean(row.get(urd_key, ""))
                if en and ur:
                    pairs.append((en, ur))
    else:
        en_file = next((p for p in root.glob("*english*") if p.suffix in {".txt", ".en"}), None)
        ur_file = next((p for p in root.glob("*urdu*")    if p.suffix in {".txt", ".ur"}), None)
        if not en_file or not ur_file:
            raise FileNotFoundError(f"No CSV or parallel txt files found in {root}")
        with open(en_file, encoding="utf-8", errors="ignore") as f:
            en = [_clean(l) for l in f if l.strip()]
        with open(ur_file, encoding="utf-8", errors="ignore") as f:
            ur = [_clean(l) for l in f if l.strip()]
        pairs = list(zip(en, ur))
    return pairs


def split_pairs(pairs, val_ratio=0.05, test_ratio=0.02, seed=42):
    rng = random.Random(seed)
    idx = list(range(len(pairs))); rng.shuffle(idx)
    n = len(pairs)
    n_val = int(n * val_ratio); n_test = int(n * test_ratio)
    test = [pairs[i] for i in idx[:n_test]]
    val = [pairs[i] for i in idx[n_test:n_test + n_val]]
    train = [pairs[i] for i in idx[n_test + n_val:]]
    return train, val, test


def train_bpe(sentences, vocab_size=8000, save_path=None):
    """Train a BPE tokenizer on an iterable of strings."""
    tok = Tokenizer(BPE(unk_token="<unk>"))
    tok.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(vocab_size=vocab_size, special_tokens=SPECIAL,
                         show_progress=False)
    tok.train_from_iterator(sentences, trainer)
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        tok.save(save_path)
    return tok


def load_bpe(path):
    return Tokenizer.from_file(path)


def encode(tok, s, add_bos=False, add_eos=False, max_len=128):
    ids = tok.encode(s).ids[: max_len - 2]
    if add_bos: ids = [BOS_IDX] + ids
    if add_eos: ids = ids + [EOS_IDX]
    return ids


class TranslationDataset(Dataset):
    def __init__(self, pairs, src_tok, tgt_tok, max_len=128):
        self.pairs = pairs
        self.src_tok = src_tok
        self.tgt_tok = tgt_tok
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        en, ur = self.pairs[i]
        src = encode(self.src_tok, en, add_bos=True, add_eos=True, max_len=self.max_len)
        tgt = encode(self.tgt_tok, ur, add_bos=True, add_eos=True, max_len=self.max_len)
        return torch.tensor(src), torch.tensor(tgt)


def collate_fn(batch):
    src, tgt = zip(*batch)
    src = pad_sequence(src, batch_first=True, padding_value=PAD_IDX)
    tgt = pad_sequence(tgt, batch_first=True, padding_value=PAD_IDX)
    return src, tgt


In [ ]:
%%writefile transformer_train.py
"""
Train the from-scratch Transformer on the English-Urdu parallel corpus.

Features:
- BPE tokenizers trained on each side.
- Noam-style LR warmup (d_model^-0.5 * min(step^-0.5, step*warmup^-1.5)).
- Label smoothing 0.1.
- Gradient clipping.
- Checkpoint every epoch + resume support.
- BLEU evaluation on a validation split using sacrebleu.

Typical run on Colab GPU:
    python transformer_train.py --data_root /content/data --epochs 30
"""
import argparse
import math
import os
import time
from pathlib import Path

import sacrebleu
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from data_utils import (BOS_IDX, EOS_IDX, PAD_IDX, TranslationDataset,
                        collate_fn, load_pairs, split_pairs, train_bpe)
from transformer_model import Transformer, count_params


def noam_lr(step, d_model, warmup):
    step = max(step, 1)
    return d_model ** -0.5 * min(step ** -0.5, step * warmup ** -1.5)


def label_smoothed_nll(logits, target, smoothing=0.1, ignore_index=PAD_IDX):
    logp = F.log_softmax(logits, dim=-1)
    nll = -logp.gather(dim=-1, index=target.unsqueeze(-1)).squeeze(-1)
    smooth = -logp.mean(dim=-1)
    loss = (1 - smoothing) * nll + smoothing * smooth
    mask = (target != ignore_index).float()
    return (loss * mask).sum() / mask.sum().clamp_min(1.0)


@torch.no_grad()
def greedy_decode(model, src, max_len, device, bos=BOS_IDX, eos=EOS_IDX):
    model.eval()
    src_mask = model.make_src_mask(src)
    enc = model.encode(src, src_mask)
    ys = torch.full((src.size(0), 1), bos, dtype=torch.long, device=device)
    for _ in range(max_len - 1):
        tgt_mask = model.make_tgt_mask(ys)
        out = model.decode(ys, enc, src_mask, tgt_mask)
        nxt = out[:, -1].argmax(-1, keepdim=True)
        ys = torch.cat([ys, nxt], dim=1)
        if (nxt == eos).all():
            break
    return ys


def detokenize(tok, ids):
    ids = [i for i in ids if i not in (PAD_IDX, BOS_IDX, EOS_IDX)]
    return tok.decode(ids)


def evaluate_bleu(model, loader, tgt_tok, device, max_len):
    hyps, refs = [], []
    for src, tgt in loader:
        src = src.to(device)
        pred = greedy_decode(model, src, max_len, device)
        for p, t in zip(pred.tolist(), tgt.tolist()):
            hyps.append(detokenize(tgt_tok, p))
            refs.append(detokenize(tgt_tok, t))
    return sacrebleu.corpus_bleu(hyps, [refs]).score, hyps[:5], refs[:5]


def train(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    weights_dir = Path(args.weights_dir); weights_dir.mkdir(parents=True, exist_ok=True)

    # --- Data ---
    pairs = load_pairs(args.data_root)
    print(f"Loaded {len(pairs)} parallel sentence pairs.")
    train_pairs, val_pairs, test_pairs = split_pairs(pairs)
    print(f"train {len(train_pairs)} / val {len(val_pairs)} / test {len(test_pairs)}")

    # --- Tokenizers ---
    src_tok_path = weights_dir / "bpe_en.json"
    tgt_tok_path = weights_dir / "bpe_ur.json"
    if src_tok_path.exists() and tgt_tok_path.exists() and args.resume:
        from data_utils import load_bpe
        src_tok, tgt_tok = load_bpe(str(src_tok_path)), load_bpe(str(tgt_tok_path))
        print("Loaded existing BPE tokenizers.")
    else:
        src_tok = train_bpe((en for en, _ in train_pairs),
                            vocab_size=args.vocab_size, save_path=str(src_tok_path))
        tgt_tok = train_bpe((ur for _, ur in train_pairs),
                            vocab_size=args.vocab_size, save_path=str(tgt_tok_path))

    src_vocab = src_tok.get_vocab_size(); tgt_vocab = tgt_tok.get_vocab_size()
    print(f"Vocab sizes: en={src_vocab}, ur={tgt_vocab}")

    train_ds = TranslationDataset(train_pairs, src_tok, tgt_tok, max_len=args.max_len)
    val_ds = TranslationDataset(val_pairs, src_tok, tgt_tok, max_len=args.max_len)
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True,
                              collate_fn=collate_fn, num_workers=args.num_workers)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False,
                            collate_fn=collate_fn, num_workers=args.num_workers)

    # --- Model ---
    model = Transformer(src_vocab, tgt_vocab,
                        d_model=args.d_model, n_heads=args.n_heads,
                        n_enc=args.n_layers, n_dec=args.n_layers,
                        d_ff=args.d_ff, dropout=args.dropout,
                        max_len=args.max_len, pad_idx=PAD_IDX).to(device)
    print(f"Model params: {count_params(model):,}")
    opt = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)

    start_epoch = 0; step = 0
    ckpt_path = weights_dir / "latest.pt"
    if args.resume and ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        start_epoch = ck["epoch"] + 1; step = ck["step"]
        print(f"Resumed from epoch {start_epoch}, step {step}")

    for epoch in range(start_epoch, args.epochs):
        t0 = time.time(); model.train(); total = 0.0; count = 0
        for i, (src, tgt) in enumerate(train_loader):
            src = src.to(device); tgt = tgt.to(device)
            tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]

            logits = model(src, tgt_in)
            loss = label_smoothed_nll(logits, tgt_out, smoothing=0.1)

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            step += 1
            for g in opt.param_groups:
                g["lr"] = args.lr_scale * noam_lr(step, args.d_model, args.warmup)
            opt.step()

            total += loss.item(); count += 1
            if i % args.log_every == 0:
                print(f"ep {epoch} it {i}/{len(train_loader)} "
                      f"loss {loss.item():.3f} lr {opt.param_groups[0]['lr']:.2e}")

        print(f"ep {epoch} avg_loss {total / max(1, count):.3f} time {time.time() - t0:.1f}s")

        # BLEU on a random 200-sample subset to stay fast
        val_loader_small = DataLoader(
            torch.utils.data.Subset(val_ds, list(range(min(200, len(val_ds))))),
            batch_size=args.batch_size, collate_fn=collate_fn)
        bleu, sample_hyps, sample_refs = evaluate_bleu(model, val_loader_small, tgt_tok,
                                                      device, args.max_len)
        print(f"ep {epoch} BLEU(val 200)={bleu:.2f}")
        for h, r in zip(sample_hyps[:3], sample_refs[:3]):
            print(f"  hyp: {h}\n  ref: {r}\n")

        torch.save({"epoch": epoch, "step": step,
                    "model": model.state_dict(), "opt": opt.state_dict(),
                    "args": vars(args),
                    "src_vocab": src_vocab, "tgt_vocab": tgt_vocab},
                   ckpt_path)
        torch.save({"epoch": epoch, "model": model.state_dict(),
                    "args": vars(args),
                    "src_vocab": src_vocab, "tgt_vocab": tgt_vocab},
                   weights_dir / f"epoch_{epoch:02d}.pt")


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--data_root", type=str, default="./data")
    p.add_argument("--weights_dir", type=str, default="./weights")
    p.add_argument("--vocab_size", type=int, default=8000)
    p.add_argument("--d_model", type=int, default=256)
    p.add_argument("--n_heads", type=int, default=8)
    p.add_argument("--n_layers", type=int, default=4)
    p.add_argument("--d_ff", type=int, default=1024)
    p.add_argument("--dropout", type=float, default=0.1)
    p.add_argument("--max_len", type=int, default=128)
    p.add_argument("--batch_size", type=int, default=32)
    p.add_argument("--num_workers", type=int, default=2)
    p.add_argument("--epochs", type=int, default=30)
    p.add_argument("--warmup", type=int, default=2000)
    p.add_argument("--lr_scale", type=float, default=1.0)
    p.add_argument("--log_every", type=int, default=50)
    p.add_argument("--resume", action="store_true")
    return p.parse_args()


if __name__ == "__main__":
    train(parse_args())


In [ ]:
%%writefile inference.py
"""
Load a trained Transformer checkpoint and translate English -> Urdu.

Usage:
    python inference.py --ckpt weights/latest.pt "How are you today?"
"""
import argparse
from pathlib import Path

import torch

from data_utils import BOS_IDX, EOS_IDX, PAD_IDX, load_bpe, encode
from transformer_model import Transformer


def load_model(ckpt_path, bpe_dir=None, device=None):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ck = torch.load(ckpt_path, map_location=device)
    a = ck["args"]
    model = Transformer(ck["src_vocab"], ck["tgt_vocab"],
                        d_model=a["d_model"], n_heads=a["n_heads"],
                        n_enc=a["n_layers"], n_dec=a["n_layers"],
                        d_ff=a["d_ff"], dropout=0.0,
                        max_len=a["max_len"], pad_idx=PAD_IDX).to(device).eval()
    model.load_state_dict(ck["model"])

    bpe_dir = Path(bpe_dir or Path(ckpt_path).parent)
    src_tok = load_bpe(str(bpe_dir / "bpe_en.json"))
    tgt_tok = load_bpe(str(bpe_dir / "bpe_ur.json"))
    return model, src_tok, tgt_tok, device, a["max_len"]


@torch.no_grad()
def translate(model, src_tok, tgt_tok, device, sentence, max_len=128):
    ids = encode(src_tok, sentence, add_bos=True, add_eos=True, max_len=max_len)
    src = torch.tensor([ids], device=device)
    src_mask = model.make_src_mask(src)
    enc = model.encode(src, src_mask)
    ys = torch.full((1, 1), BOS_IDX, dtype=torch.long, device=device)
    for _ in range(max_len - 1):
        tgt_mask = model.make_tgt_mask(ys)
        out = model.decode(ys, enc, src_mask, tgt_mask)
        nxt = out[:, -1].argmax(-1, keepdim=True)
        ys = torch.cat([ys, nxt], 1)
        if nxt.item() == EOS_IDX:
            break
    ids = [i for i in ys[0].tolist() if i not in (PAD_IDX, BOS_IDX, EOS_IDX)]
    return tgt_tok.decode(ids)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--ckpt", type=str, default="./weights/latest.pt")
    p.add_argument("sentences", nargs="+")
    args = p.parse_args()

    model, src_tok, tgt_tok, device, max_len = load_model(args.ckpt)
    for s in args.sentences:
        print(f"EN: {s}")
        print(f"UR: {translate(model, src_tok, tgt_tok, device, s, max_len)}")
        print()


if __name__ == "__main__":
    main()


In [ ]:
# Install NLP deps not already on Kaggle images
!pip -q install tokenizers sacrebleu

In [ ]:
# Train the from-scratch Transformer. ~20-25 min on T4 x2.
!python transformer_train.py \
   --data_root /kaggle/working/data \
   --weights_dir /kaggle/working/weights \
   --epochs 20 --batch_size 64 --d_model 256 --n_heads 8 --n_layers 4 --d_ff 1024 \
   --warmup 2000 --resume

In [ ]:
# Qualitative translation samples (screenshot for your report)
!python inference.py --ckpt /kaggle/working/weights/latest.pt \
   "How are you today?" \
   "I love reading books in the library." \
   "The weather is very nice this morning." \
   "She is a very good teacher." \
   "Education is the key to success."

In [ ]:
# Zip weights so you can download them from the Output panel (right sidebar)
!tar -czf /kaggle/working/q2_weights.tar.gz -C /kaggle/working weights
!ls -lh /kaggle/working/q2_weights.tar.gz